# Sensitivity Analysis

This notebook investigates how sensitive the anomaly detection results are
to key hyperparameters. Robustness across parameter choices strengthens
the paper's conclusions.

Parameters varied:
1. **PCA components** — how dimensionality reduction affects KS Test scores
2. **KNN neighbourhood size** — how K affects KS Test and Wasserstein scores
3. **Contamination rate** — how the assumed anomaly fraction affects flagging

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
from config import (
    GENRES, RESULTS_DIR, RANDOM_STATE,
    genre_dataset_dir, genre_results_dir,
)

OUT_DIR = RESULTS_DIR / 'sensitivity_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUT_DIR}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import ks_2samp, wasserstein_distance
from tqdm import tqdm

In [ ]:
# ── Load raw embeddings for all genres ──

data = {}
for genre in GENRES:
    d_dir = genre_dataset_dir(genre, 'injected')
    emb_raw = np.load(d_dir / 'embeddings.npy')
    meta    = pd.read_csv(d_dir / 'metadata.csv')
    data[genre] = {'embeddings_raw': emb_raw, 'metadata': meta}
    y = meta['is_anomaly'].values
    print(f'{genre:<15} {emb_raw.shape[0]:>5} paintings | {y.sum()} anomalies | dim={emb_raw.shape[1]}')

## Helper Functions

Replicating the scoring logic from the individual analysis notebooks
so we can re-compute scores with different parameter settings.

In [ ]:
def cosine_anomaly_scores(embeddings_raw):
    """Phase 1 global cosine similarity (works on raw 2048-dim embeddings)."""
    emb = normalize(embeddings_raw, norm='l2')
    centroid = normalize(emb.mean(axis=0, keepdims=True), norm='l2')
    return 1 - cosine_similarity(emb, centroid).flatten()


def bh_correction(p_values, alpha=0.05):
    """Benjamini-Hochberg FDR correction."""
    p = np.asarray(p_values)
    n = len(p)
    sorted_idx = np.argsort(p)
    sorted_p   = p[sorted_idx]
    thresholds = (np.arange(1, n + 1) / n) * alpha
    below = sorted_p <= thresholds
    if below.any():
        cutoff = np.where(below)[0][-1]
        below[:cutoff + 1] = True
    reject = np.zeros(n, dtype=bool)
    reject[sorted_idx] = below
    return reject


def ks_painting_scores(embeddings_pca, knn_k=20):
    """Phase 2 per-painting KS test scores."""
    nn = NearestNeighbors(n_neighbors=knn_k + 1, metric='cosine', n_jobs=-1)
    nn.fit(embeddings_pca)
    _, knn_idx = nn.kneighbors(embeddings_pca)
    knn_idx = knn_idx[:, 1:]  # exclude self

    n = len(embeddings_pca)
    n_dims = embeddings_pca.shape[1]
    mean_d = np.zeros(n)

    for i in range(n):
        neighbourhood = embeddings_pca[knn_idx[i]]
        d_stats = np.zeros(n_dims)
        for dim in range(n_dims):
            d_stats[dim], _ = ks_2samp(neighbourhood[:, dim], embeddings_pca[:, dim])
        mean_d[i] = d_stats.mean()

    return mean_d


def swd_painting_scores(embeddings_pca, knn_k=20, n_projections=100):
    """Phase 2 per-painting Sliced Wasserstein Distance."""
    rng = np.random.RandomState(RANDOM_STATE)
    d = embeddings_pca.shape[1]
    directions = rng.randn(n_projections, d)
    directions /= np.linalg.norm(directions, axis=1, keepdims=True)

    nn = NearestNeighbors(n_neighbors=knn_k + 1, metric='cosine', n_jobs=-1)
    nn.fit(embeddings_pca)
    _, knn_idx = nn.kneighbors(embeddings_pca)
    knn_idx = knn_idx[:, 1:]

    # Pre-project the full dataset
    Y_proj = embeddings_pca @ directions.T  # (N, n_proj)

    n = len(embeddings_pca)
    swd_scores = np.zeros(n)

    for i in range(n):
        neighbourhood = embeddings_pca[knn_idx[i]]
        X_proj = neighbourhood @ directions.T  # (K, n_proj)
        swd_scores[i] = np.mean([
            wasserstein_distance(X_proj[:, j], Y_proj[:, j])
            for j in range(n_projections)
        ])

    return swd_scores


print('Helper functions defined.')

## 1. Sensitivity to PCA Dimensions

The default is 50 PCA components. We test how detection performance
changes with different dimensionalities.

**Note**: Cosine Similarity uses raw embeddings (2048-d) — no PCA. So it
serves as a stable baseline here.

In [ ]:
PCA_DIMS = [10, 20, 30, 50, 75, 100, 150]

pca_results = []

for genre in GENRES:
    emb_raw = data[genre]['embeddings_raw']
    y       = data[genre]['metadata']['is_anomaly'].values
    emb_l2  = normalize(emb_raw, norm='l2')

    # Cosine baseline (constant across PCA dims)
    cosine_scores = cosine_anomaly_scores(emb_raw)
    cosine_auc = roc_auc_score(y, cosine_scores)

    for n_comp in tqdm(PCA_DIMS, desc=genre):
        pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
        emb_pca = pca.fit_transform(emb_l2)
        var_explained = pca.explained_variance_ratio_.sum()

        # KS Test
        ks_scores = ks_painting_scores(emb_pca, knn_k=20)
        ks_auc = roc_auc_score(y, ks_scores)

        pca_results.append({
            'genre': genre, 'n_pca': n_comp,
            'var_explained': var_explained,
            'Cosine AUC': cosine_auc,
            'KS Test AUC': ks_auc,
        })
        print(f'  {genre} | PCA={n_comp:>3} | var={var_explained:.3f} | KS AUC={ks_auc:.4f}')

pca_df = pd.DataFrame(pca_results)
pca_df.to_csv(OUT_DIR / 'pca_sensitivity.csv', index=False)
print('\nDone.')

In [ ]:
fig, axes = plt.subplots(1, len(GENRES), figsize=(6 * len(GENRES), 5), sharey=True)

for ax, genre in zip(axes, GENRES):
    sub = pca_df[pca_df['genre'] == genre]
    ax.plot(sub['n_pca'], sub['KS Test AUC'], 'o-', color='forestgreen',
            linewidth=2, markersize=6, label='KS Test')

    # Cosine baseline
    cosine_val = sub['Cosine AUC'].iloc[0]
    ax.axhline(cosine_val, color='steelblue', linestyle='--', linewidth=1.5,
               label=f'Cosine Sim ({cosine_val:.3f})')

    # Random baseline
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)

    ax.set_xlabel('PCA Components')
    if ax is axes[0]:
        ax.set_ylabel('AUC-ROC')
    ax.set_title(genre.title(), fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(0.4, 1.0)

# Variance explained on secondary y-axis (last plot)
ax2 = axes[-1].twinx()
sub = pca_df[pca_df['genre'] == GENRES[-1]]
ax2.bar(sub['n_pca'], sub['var_explained'], alpha=0.15, color='gray', width=5)
ax2.set_ylabel('Cumulative Variance Explained', fontsize=9, color='gray')
ax2.set_ylim(0, 1.05)

fig.suptitle('Effect of PCA Dimensionality on Detection Performance',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_sensitivity.png', dpi=150)
plt.show()

## 2. Sensitivity to KNN Neighbourhood Size (K)

KS Test and Wasserstein both compare each painting's K-nearest-neighbour
distribution to the global distribution. How does K affect results?

In [ ]:
K_VALUES = [5, 10, 15, 20, 30, 50, 75]

knn_results = []

for genre in GENRES:
    emb_raw = data[genre]['embeddings_raw']
    y       = data[genre]['metadata']['is_anomaly'].values
    emb_l2  = normalize(emb_raw, norm='l2')

    # PCA with default 50 components
    pca = PCA(n_components=50, random_state=RANDOM_STATE)
    emb_pca = pca.fit_transform(emb_l2)

    # Cosine baseline
    cosine_auc = roc_auc_score(y, cosine_anomaly_scores(emb_raw))

    for k in tqdm(K_VALUES, desc=genre):
        ks_scores = ks_painting_scores(emb_pca, knn_k=k)
        ks_auc    = roc_auc_score(y, ks_scores)

        knn_results.append({
            'genre': genre, 'K': k,
            'Cosine AUC': cosine_auc,
            'KS Test AUC': ks_auc,
        })
        print(f'  {genre} | K={k:>3} | KS AUC={ks_auc:.4f}')

knn_df = pd.DataFrame(knn_results)
knn_df.to_csv(OUT_DIR / 'knn_k_sensitivity.csv', index=False)
print('\nDone.')

In [ ]:
fig, axes = plt.subplots(1, len(GENRES), figsize=(6 * len(GENRES), 5), sharey=True)

for ax, genre in zip(axes, GENRES):
    sub = knn_df[knn_df['genre'] == genre]
    ax.plot(sub['K'], sub['KS Test AUC'], 's-', color='forestgreen',
            linewidth=2, markersize=6, label='KS Test')

    cosine_val = sub['Cosine AUC'].iloc[0]
    ax.axhline(cosine_val, color='steelblue', linestyle='--', linewidth=1.5,
               label=f'Cosine Sim ({cosine_val:.3f})')
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)

    # Mark default K=20
    default_row = sub[sub['K'] == 20]
    if not default_row.empty:
        ax.axvline(20, color='red', linestyle='--', linewidth=1, alpha=0.4,
                   label='Default K=20')

    ax.set_xlabel('K (neighbourhood size)')
    if ax is axes[0]:
        ax.set_ylabel('AUC-ROC')
    ax.set_title(genre.title(), fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(0.4, 1.0)

fig.suptitle('Effect of KNN Neighbourhood Size on Detection Performance',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'knn_k_sensitivity.png', dpi=150)
plt.show()

## 3. Sensitivity to Contamination Rate

The default dataset has ~5% anomalies (75 out of ~1500). If the pipeline
were deployed with a different assumed contamination rate, how would
detection performance (flagging precision/recall) change?

Note: AUC-ROC is **threshold-free** — it doesn't depend on the contamination
assumption. However, Precision@K and F1 at a fixed threshold do.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

CONTAMINATION_RATES = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20]

contam_results = []

for genre in GENRES:
    y  = data[genre]['metadata']['is_anomaly'].values
    n  = len(y)

    # Load pre-computed scores (from results files)
    r_dir = genre_results_dir(genre, 'injected')
    cosine_df = pd.read_csv(r_dir / 'phase1_global_cosine_scores.csv')
    ks_df     = pd.read_csv(r_dir / 'ks_phase2_painting_scores.csv')

    score_sets = {
        'Cosine Similarity': cosine_df['global_anomaly_score'].values,
        'KS Test':           ks_df['ks_mean_d'].values,
    }

    for method, scores in score_sets.items():
        for c_rate in CONTAMINATION_RATES:
            k = max(1, int(n * c_rate))  # flag top-k
            threshold = np.sort(scores)[::-1][k - 1]  # k-th highest score
            y_pred = (scores >= threshold).astype(int)

            prec = precision_score(y, y_pred, zero_division=0)
            rec  = recall_score(y, y_pred, zero_division=0)
            f1   = f1_score(y, y_pred, zero_division=0)

            contam_results.append({
                'genre': genre, 'method': method,
                'contamination': c_rate, 'k_flagged': k,
                'precision': prec, 'recall': rec, 'f1': f1,
            })

contam_df = pd.DataFrame(contam_results)
contam_df.to_csv(OUT_DIR / 'contamination_sensitivity.csv', index=False)

# Show the table
for method in ['Cosine Similarity', 'KS Test']:
    print(f'\n=== {method} ===\n')
    sub = contam_df[contam_df['method'] == method]
    piv = sub.pivot_table(index='contamination', columns='genre',
                          values='f1', aggfunc='first').round(4)
    piv['Mean F1'] = piv.mean(axis=1)
    print(piv.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['precision', 'recall', 'f1']
metric_names = ['Precision', 'Recall', 'F1 Score']

for ax, metric, mname in zip(axes, metrics, metric_names):
    for method in ['Cosine Similarity', 'KS Test']:
        sub = contam_df[contam_df['method'] == method]
        agg = sub.groupby('contamination')[metric].mean()
        style = '-o' if method == 'Cosine Similarity' else '--s'
        color = 'steelblue' if method == 'Cosine Similarity' else 'forestgreen'
        ax.plot(agg.index, agg.values, style, color=color,
                linewidth=2, markersize=6, label=method)

    # Mark true contamination rate
    ax.axvline(0.05, color='red', linestyle='--', linewidth=1, alpha=0.5,
               label='True rate (5%)')
    ax.set_xlabel('Assumed Contamination Rate')
    ax.set_ylabel(mname)
    ax.set_title(mname, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.05)

fig.suptitle('Effect of Contamination Rate Assumption on Detection Quality\n(averaged across genres)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'contamination_sensitivity.png', dpi=150)
plt.show()

## 4. Summary Table

Consolidated sensitivity results for the paper.

In [ ]:
# PCA sensitivity summary
print('=== PCA Dimensionality Sensitivity (KS Test, Mean AUC across genres) ===\n')
pca_summary = pca_df.groupby('n_pca')['KS Test AUC'].agg(['mean', 'std']).round(4)
pca_summary.columns = ['Mean AUC', 'Std AUC']
print(pca_summary.to_string())

best_pca = pca_summary['Mean AUC'].idxmax()
print(f'\nBest PCA dimensions: {best_pca} (AUC = {pca_summary.loc[best_pca, "Mean AUC"]:.4f})')
print(f'Default (50):          AUC = {pca_summary.loc[50, "Mean AUC"]:.4f}')

print('\n\n=== KNN K Sensitivity (KS Test, Mean AUC across genres) ===\n')
knn_summary = knn_df.groupby('K')['KS Test AUC'].agg(['mean', 'std']).round(4)
knn_summary.columns = ['Mean AUC', 'Std AUC']
print(knn_summary.to_string())

best_k = knn_summary['Mean AUC'].idxmax()
print(f'\nBest K: {best_k} (AUC = {knn_summary.loc[best_k, "Mean AUC"]:.4f})')
print(f'Default (20):   AUC = {knn_summary.loc[20, "Mean AUC"]:.4f}')

print('\n\n=== Contamination Rate Sensitivity (Cosine Similarity, Mean F1) ===\n')
cos_contam = contam_df[contam_df['method'] == 'Cosine Similarity']
f1_summary = cos_contam.groupby('contamination')['f1'].mean().round(4)
print(f1_summary.to_string())
print(f'\nBest contamination rate: {f1_summary.idxmax()} (F1 = {f1_summary.max():.4f})')